# CartFlow (P06) — Week 5: Silver Candidate Transformation

Based on the approved P06 CartFlow Student Project Playbook and Week-5 conversion
requirements. Table names, columns, and standardisation domains are CartFlow-specific.

**Approved observable objectives:**
- Cast timestamps and numeric fields safely.
- Standardise bounded categories and identifiers.
- Derive only the approved delivery and freight fields.
- Retain Bronze lineage in Candidate.
- Prove Bronze-to-Candidate reconciliation.

**Expected result (per playbook):** five typed Silver Candidate tables that retain all
physical records and expose parse/standardisation issues for Week 6 Data Quality — no
rows are filtered, deduplicated or quarantined here.

# Part 1 — Confirm the Week-4 Bronze handoff

Verify the catalog/schema and that all five approved Bronze inputs exist before any
transformation is written.

In [ ]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog, current_schema() AS active_schema;

In [ ]:
%sql
SHOW TABLES LIKE 'bronze_ecommerce_*';

**Expected result:** all five Bronze tables are listed —
`bronze_ecommerce_orders`, `bronze_ecommerce_order_items`, `bronze_ecommerce_payments`,
`bronze_ecommerce_reviews`, `bronze_ecommerce_sellers`. Stop and complete Week 4 first if
any is missing.

### Record the actual Bronze schema and starting counts

In [ ]:
%sql
DESCRIBE TABLE bronze_ecommerce_orders;

In [ ]:
%sql
DESCRIBE TABLE bronze_ecommerce_order_items;

In [ ]:
%sql
DESCRIBE TABLE bronze_ecommerce_payments;

In [ ]:
%sql
DESCRIBE TABLE bronze_ecommerce_reviews;

In [ ]:
%sql
DESCRIBE TABLE bronze_ecommerce_sellers;

In [ ]:
%sql
SELECT 'orders' AS entity, COUNT(*) AS bronze_rows FROM bronze_ecommerce_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM bronze_ecommerce_order_items
UNION ALL
SELECT 'payments', COUNT(*) FROM bronze_ecommerce_payments
UNION ALL
SELECT 'reviews', COUNT(*) FROM bronze_ecommerce_reviews
UNION ALL
SELECT 'sellers', COUNT(*) FROM bronze_ecommerce_sellers;

**Record the actual counts from your own run here** — never copy a count from a
worked example.

# Part 2 — Transformation contract

Column-by-column contract for the CartFlow Week-5 Silver Candidate layer. The approved playbook requires typed Candidate tables, safe casting, domain standardisation, approved derived fields, and retained Bronze lineage.

| Entity | Standardise | Type safely (`try_cast`) | Derived fields |
|---|---|---|---|
| `orders` | `source_record_id`, `order_id`, `customer_city_code`, `customer_region`, `customer_state`, `currency_code` → TRIM+UPPER; `order_status`, `customer_segment` → TRIM+LOWER | `purchase_ts`, `approval_ts`, `carrier_handoff_ts`, `delivered_ts`, `estimated_delivery_ts`, `return_ts` → TIMESTAMP | `delivery_days`, `delay_days`, `delivery_band` |
| `order_items` | `source_record_id`, `order_item_id`, `order_id`, `product_id`, `seller_id` → TRIM+UPPER; `category_code` → TRIM+LOWER | `item_price`, `freight_value` → DECIMAL(12,2); `quantity` → INT; `item_created_ts` → TIMESTAMP | `item_total`, `freight_share` |
| `payments` | `source_record_id`, `payment_id`, `order_id`, `currency_code` → TRIM+UPPER; `payment_method` → TRIM+LOWER | `installment_no` → INT; `payment_value` → DECIMAL(12,2); `payment_ts` → TIMESTAMP | none documented |
| `reviews` | `source_record_id`, `review_id`, `order_id` → TRIM+UPPER; `review_sentiment` → TRIM+LOWER | `review_score` → INT; `review_date` → TIMESTAMP | none documented |
| `sellers` | `source_record_id`, `seller_id`, `seller_region`, `seller_state` → TRIM+UPPER; `seller_type`, `service_band`, `seller_status` → controlled domain casing | `active_from`, `active_to` → DATE | none documented |

All entities retain Bronze lineage (`_source_file_name`, `_source_file_path`, `_ingested_at`, `_ingestion_run_id`, `_bronze_schema_version`, `_bronze_record_hash`, `_rescued_payload`) and add Candidate audit fields (`_candidate_created_at`, `_candidate_schema_version`).

> **Formula authority note:** the approved Week-5 playbook names `freight_share` and `delivery_band` as required derived fields, but the excerpt available here does not specify their exact formulas/bucket boundaries. The implementation below keeps the formulas explicit and visible for mentor/data-dictionary confirmation rather than hiding the assumption.


# Part 3 — Build `silver_candidate_orders`

### A. Standardise

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_standardised AS
SELECT
  upper(trim(source_record_id)) AS source_record_id,
  upper(trim(order_id)) AS order_id,
  upper(trim(customer_region)) AS customer_region,
  upper(trim(customer_state)) AS customer_state,
  upper(trim(customer_city_code)) AS customer_city_code,
  lower(trim(customer_segment)) AS customer_segment,
  lower(trim(order_status)) AS order_status,
  purchase_ts, approval_ts, carrier_handoff_ts, delivered_ts,
  estimated_delivery_ts, return_ts,
  upper(trim(currency_code)) AS currency_code,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash, _rescued_payload
FROM bronze_ecommerce_orders;

### B. Type safely

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_typed AS
SELECT
  source_record_id, order_id, customer_region, customer_state,
  customer_city_code, customer_segment, order_status,
  try_cast(purchase_ts AS TIMESTAMP) AS purchase_ts,
  try_cast(approval_ts AS TIMESTAMP) AS approval_ts,
  try_cast(carrier_handoff_ts AS TIMESTAMP) AS carrier_handoff_ts,
  try_cast(delivered_ts AS TIMESTAMP) AS delivered_ts,
  try_cast(estimated_delivery_ts AS TIMESTAMP) AS estimated_delivery_ts,
  try_cast(return_ts AS TIMESTAMP) AS return_ts,
  currency_code,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id,
  _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash,
  _rescued_payload
FROM orders_standardised;

In [ ]:
%sql
DESCRIBE orders_typed;

### C. Calculate — `delivery_days`, `delay_days`, `delivery_band`

Only computed where both required inputs are non-null (per the playbook: "derive ...
only where inputs are valid").

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW orders_candidate_ready AS
SELECT
  *,
  CASE WHEN purchase_ts IS NOT NULL AND delivered_ts IS NOT NULL
       THEN datediff(to_date(delivered_ts), to_date(purchase_ts)) END AS delivery_days,
  CASE WHEN estimated_delivery_ts IS NOT NULL AND delivered_ts IS NOT NULL
       THEN datediff(to_date(delivered_ts), to_date(estimated_delivery_ts)) END AS delay_days,
  CASE
    WHEN estimated_delivery_ts IS NULL OR delivered_ts IS NULL THEN NULL
    WHEN datediff(to_date(delivered_ts), to_date(estimated_delivery_ts)) <= 0 THEN 'early_or_on_time'
    WHEN datediff(to_date(delivered_ts), to_date(estimated_delivery_ts)) BETWEEN 1 AND 3 THEN 'late_1_3_days'
    ELSE 'late_4_plus_days'
  END AS delivery_band,
  current_timestamp() AS _candidate_created_at,
  'ecommerce_candidate_v1.0' AS _candidate_schema_version
FROM orders_typed;

### D. Inspect — sample and conversion visibility

In [ ]:
%sql
SELECT order_id, order_status, purchase_ts, delivered_ts, estimated_delivery_ts,
       delivery_days, delay_days, delivery_band
FROM orders_candidate_ready
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN purchase_ts IS NOT NULL AND try_cast(purchase_ts AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS purchase_ts_parse_failures,
  SUM(CASE WHEN approval_ts IS NOT NULL AND try_cast(approval_ts AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS approval_ts_parse_failures,
  SUM(CASE WHEN delivered_ts IS NOT NULL AND try_cast(delivered_ts AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS delivered_ts_parse_failures,
  SUM(CASE WHEN estimated_delivery_ts IS NOT NULL AND try_cast(estimated_delivery_ts AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS estimated_delivery_ts_parse_failures,
  SUM(CASE WHEN return_ts IS NOT NULL AND try_cast(return_ts AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS return_ts_parse_failures
FROM bronze_ecommerce_orders;

**Expected result:** actual failure counts from your data. A non-zero value is evidence
for Week 6 — it is not corrected here.

### E. Persist

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_orders
USING DELTA
AS SELECT * FROM orders_candidate_ready;

# Part 4 — Build `silver_candidate_order_items`

### A. Standardise

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW order_items_standardised AS
SELECT
  upper(trim(source_record_id)) AS source_record_id,
  upper(trim(order_item_id)) AS order_item_id,
  upper(trim(order_id)) AS order_id,
  order_item_seq,
  upper(trim(product_id)) AS product_id,
  lower(trim(category_code)) AS category_code,
  upper(trim(seller_id)) AS seller_id,
  item_price, freight_value, quantity, item_created_ts,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash, _rescued_payload
FROM bronze_ecommerce_order_items;

### B. Type safely

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW order_items_typed AS
SELECT
  source_record_id, order_item_id, order_id,
  try_cast(order_item_seq AS INT) AS order_item_seq,
  product_id, category_code, seller_id,
  try_cast(item_price AS DECIMAL(12,2)) AS item_price,
  try_cast(freight_value AS DECIMAL(12,2)) AS freight_value,
  try_cast(quantity AS INT) AS quantity,
  try_cast(item_created_ts AS TIMESTAMP) AS item_created_ts,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id,
  _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash,
  _rescued_payload
FROM order_items_standardised;

In [ ]:
%sql
DESCRIBE order_items_typed;

### C. Calculate — `item_total`, `freight_share`

`item_total = item_price * quantity`. `freight_share = freight_value / item_total`
(flagged in Part 2 as an assumption pending confirmation).

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW order_items_candidate_ready AS
SELECT
  *,
  CASE WHEN item_price IS NOT NULL AND quantity IS NOT NULL
       THEN cast(item_price * quantity AS DECIMAL(12,2)) END AS item_total,
  CASE WHEN freight_value IS NOT NULL AND item_price IS NOT NULL AND quantity IS NOT NULL
            AND (item_price * quantity) <> 0
       THEN cast(freight_value / (item_price * quantity) AS DECIMAL(12,4)) END AS freight_share,
  current_timestamp() AS _candidate_created_at,
  'ecommerce_candidate_v1.0' AS _candidate_schema_version
FROM order_items_typed;

### D. Inspect — sample and conversion visibility

In [ ]:
%sql
SELECT order_item_id, order_id, category_code, item_price, quantity, freight_value,
       item_total, freight_share
FROM order_items_candidate_ready
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN item_price IS NOT NULL AND try_cast(item_price AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END) AS item_price_parse_failures,
  SUM(CASE WHEN freight_value IS NOT NULL AND try_cast(freight_value AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END) AS freight_value_parse_failures,
  SUM(CASE WHEN quantity IS NOT NULL AND try_cast(quantity AS INT) IS NULL THEN 1 ELSE 0 END) AS quantity_parse_failures
FROM bronze_ecommerce_order_items;

### E. Persist

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_order_items
USING DELTA
AS SELECT * FROM order_items_candidate_ready;

# Part 5 — Build `silver_candidate_payments`

No derived fields are documented for payments in the approved build flow — standardise,
type and retain lineage only.

### A. Standardise

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_standardised AS
SELECT
  upper(trim(source_record_id)) AS source_record_id,
  upper(trim(payment_id)) AS payment_id,
  upper(trim(order_id)) AS order_id,
  installment_no,
  lower(trim(payment_method)) AS payment_method,
  payment_value, payment_ts,
  upper(trim(currency_code)) AS currency_code,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash, _rescued_payload
FROM bronze_ecommerce_payments;

### B. Type safely

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW payments_typed AS
SELECT
  source_record_id, payment_id, order_id,
  try_cast(installment_no AS INT) AS installment_no,
  payment_method,
  try_cast(payment_value AS DECIMAL(12,2)) AS payment_value,
  try_cast(payment_ts AS TIMESTAMP) AS payment_ts,
  currency_code,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id,
  _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash,
  _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'ecommerce_candidate_v1.0' AS _candidate_schema_version
FROM payments_standardised;

In [ ]:
%sql
DESCRIBE payments_typed;

### C/D. Inspect — sample and conversion visibility

In [ ]:
%sql
SELECT payment_id, order_id, payment_method, payment_value, payment_ts, installment_no
FROM payments_typed
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN payment_value IS NOT NULL AND try_cast(payment_value AS DECIMAL(12,2)) IS NULL THEN 1 ELSE 0 END) AS payment_value_parse_failures,
  SUM(CASE WHEN installment_no IS NOT NULL AND try_cast(installment_no AS INT) IS NULL THEN 1 ELSE 0 END) AS installment_no_parse_failures,
  SUM(CASE WHEN payment_ts IS NOT NULL AND try_cast(payment_ts AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS payment_ts_parse_failures
FROM bronze_ecommerce_payments;

### E. Persist

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_payments
USING DELTA
AS SELECT * FROM payments_typed;

# Part 6 — Build `silver_candidate_reviews`

No review-specific derived field is named in the approved build-flow step list —
standardise, type and retain lineage only.

### A. Standardise

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_standardised AS
SELECT
  upper(trim(source_record_id)) AS source_record_id,
  upper(trim(review_id)) AS review_id,
  upper(trim(order_id)) AS order_id,
  review_score, review_date,
  lower(trim(review_sentiment)) AS review_sentiment,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash, _rescued_payload
FROM bronze_ecommerce_reviews;

### B. Type safely

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW reviews_typed AS
SELECT
  source_record_id, review_id, order_id,
  try_cast(review_score AS INT) AS review_score,
  try_cast(review_date AS TIMESTAMP) AS review_date,
  review_sentiment,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id,
  _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash,
  _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'ecommerce_candidate_v1.0' AS _candidate_schema_version
FROM reviews_standardised;

In [ ]:
%sql
DESCRIBE reviews_typed;

### C/D. Inspect — sample and conversion visibility

In [ ]:
%sql
SELECT review_id, order_id, review_score, review_date, review_sentiment
FROM reviews_typed
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN review_score IS NOT NULL AND try_cast(review_score AS INT) IS NULL THEN 1 ELSE 0 END) AS review_score_parse_failures,
  SUM(CASE WHEN review_date IS NOT NULL AND try_cast(review_date AS TIMESTAMP) IS NULL THEN 1 ELSE 0 END) AS review_date_parse_failures
FROM bronze_ecommerce_reviews;

### E. Persist

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_reviews
USING DELTA
AS SELECT * FROM reviews_typed;

# Part 7 — Build `silver_candidate_sellers`

No seller-specific derived field is named in the approved build-flow step list —
standardise, type and retain lineage only.

### A. Standardise

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sellers_standardised AS
SELECT
  upper(trim(source_record_id)) AS source_record_id,
  upper(trim(seller_id)) AS seller_id,
  initcap(trim(seller_region)) AS seller_region,
  upper(trim(seller_state)) AS seller_state,
  CASE upper(trim(seller_type))
    WHEN 'INDIVIDUAL' THEN 'Individual'
    WHEN 'SME' THEN 'SME'
    WHEN 'ENTERPRISE' THEN 'Enterprise'
    ELSE trim(seller_type)
  END AS seller_type,
  CASE upper(trim(service_band))
    WHEN 'STANDARD' THEN 'Standard'
    WHEN 'PRIORITY' THEN 'Priority'
    WHEN 'PREMIUM' THEN 'Premium'
    ELSE trim(service_band)
  END AS service_band,
  active_from, active_to,
  lower(trim(seller_status)) AS seller_status,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id, _schema_version, _record_hash, _rescued_payload
FROM bronze_ecommerce_sellers;


### B. Type safely

In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW sellers_typed AS
SELECT
  source_record_id, seller_id, seller_region, seller_state, seller_type, service_band,
  try_cast(active_from AS DATE) AS active_from,
  try_cast(active_to AS DATE) AS active_to,
  seller_status,
  _source_file_name, _source_file_path, _ingested_at,
  _ingestion_run_id,
  _schema_version AS _bronze_schema_version,
  _record_hash AS _bronze_record_hash,
  _rescued_payload,
  current_timestamp() AS _candidate_created_at,
  'ecommerce_candidate_v1.0' AS _candidate_schema_version
FROM sellers_standardised;

In [ ]:
%sql
DESCRIBE sellers_typed;

### C/D. Inspect — sample and conversion visibility

In [ ]:
%sql
SELECT seller_id, seller_type, service_band, active_from, active_to, seller_status
FROM sellers_typed
LIMIT 10;

In [ ]:
%sql
SELECT
  SUM(CASE WHEN active_from IS NOT NULL AND try_cast(active_from AS DATE) IS NULL THEN 1 ELSE 0 END) AS active_from_parse_failures,
  SUM(CASE WHEN active_to IS NOT NULL AND try_cast(active_to AS DATE) IS NULL THEN 1 ELSE 0 END) AS active_to_parse_failures
FROM bronze_ecommerce_sellers;

### E. Persist

In [ ]:
%sql
CREATE OR REPLACE TABLE silver_candidate_sellers
USING DELTA
AS SELECT * FROM sellers_typed;

# Part 8 — Validate the complete Week-5 output

### 8.1 Bronze-to-Candidate reconciliation — all five entities

In [ ]:
%sql
WITH counts AS (
  SELECT 'orders' AS entity,
         (SELECT COUNT(*) FROM bronze_ecommerce_orders) AS bronze_rows,
         (SELECT COUNT(*) FROM silver_candidate_orders) AS candidate_rows
  UNION ALL
  SELECT 'order_items',
         (SELECT COUNT(*) FROM bronze_ecommerce_order_items),
         (SELECT COUNT(*) FROM silver_candidate_order_items)
  UNION ALL
  SELECT 'payments',
         (SELECT COUNT(*) FROM bronze_ecommerce_payments),
         (SELECT COUNT(*) FROM silver_candidate_payments)
  UNION ALL
  SELECT 'reviews',
         (SELECT COUNT(*) FROM bronze_ecommerce_reviews),
         (SELECT COUNT(*) FROM silver_candidate_reviews)
  UNION ALL
  SELECT 'sellers',
         (SELECT COUNT(*) FROM bronze_ecommerce_sellers),
         (SELECT COUNT(*) FROM silver_candidate_sellers)
)
SELECT *, candidate_rows - bronze_rows AS difference,
       CASE WHEN candidate_rows = bronze_rows THEN 'PASS' ELSE 'CHECK' END AS status
FROM counts
ORDER BY entity;

**Expected result:** `difference = 0` and `status = PASS` for all five entities. A
mismatch means a row was lost or added — investigate before continuing; do not filter,
deduplicate or force the count to match.

### 8.1b Key uniqueness / approved grain

Week 5 must not introduce duplication or grain change. This validates the approved primary/physical business keys without filtering or deduplicating any records.


In [ ]:
%sql
-- 8.1b Key uniqueness / approved grain validation
WITH key_checks AS (
  SELECT 'orders' AS entity, COUNT(*) AS rows, COUNT(DISTINCT order_id) AS distinct_key_rows,
         CASE WHEN COUNT(*) = COUNT(DISTINCT order_id) THEN 'PASS' ELSE 'CHECK' END AS status
  FROM silver_candidate_orders
  UNION ALL
  SELECT 'order_items', COUNT(*), COUNT(DISTINCT order_item_id),
         CASE WHEN COUNT(*) = COUNT(DISTINCT order_item_id) THEN 'PASS' ELSE 'CHECK' END
  FROM silver_candidate_order_items
  UNION ALL
  SELECT 'payments', COUNT(*), COUNT(DISTINCT payment_id),
         CASE WHEN COUNT(*) = COUNT(DISTINCT payment_id) THEN 'PASS' ELSE 'CHECK' END
  FROM silver_candidate_payments
  UNION ALL
  SELECT 'reviews', COUNT(*), COUNT(DISTINCT review_id),
         CASE WHEN COUNT(*) = COUNT(DISTINCT review_id) THEN 'PASS' ELSE 'CHECK' END
  FROM silver_candidate_reviews
  UNION ALL
  SELECT 'sellers', COUNT(*), COUNT(DISTINCT seller_id),
         CASE WHEN COUNT(*) = COUNT(DISTINCT seller_id) THEN 'PASS' ELSE 'CHECK' END
  FROM silver_candidate_sellers
)
SELECT *, rows - distinct_key_rows AS duplicate_key_rows
FROM key_checks
ORDER BY entity;


### 8.2 Row-level lineage proof — all five entities

In [ ]:
%sql
-- Row-level lineage: orders
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_ecommerce_orders
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_candidate_orders
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_candidate_orders
     EXCEPT ALL
     SELECT _record_hash FROM bronze_ecommerce_orders
  )) AS unexpected_candidate_rows;

In [ ]:
%sql
-- Row-level lineage: order_items
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_ecommerce_order_items
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_candidate_order_items
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_candidate_order_items
     EXCEPT ALL
     SELECT _record_hash FROM bronze_ecommerce_order_items
  )) AS unexpected_candidate_rows;

In [ ]:
%sql
-- Row-level lineage: payments
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_ecommerce_payments
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_candidate_payments
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_candidate_payments
     EXCEPT ALL
     SELECT _record_hash FROM bronze_ecommerce_payments
  )) AS unexpected_candidate_rows;

In [ ]:
%sql
-- Row-level lineage: reviews
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_ecommerce_reviews
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_candidate_reviews
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_candidate_reviews
     EXCEPT ALL
     SELECT _record_hash FROM bronze_ecommerce_reviews
  )) AS unexpected_candidate_rows;

In [ ]:
%sql
-- Row-level lineage: sellers
SELECT
  (SELECT COUNT(*) FROM (
     SELECT _record_hash FROM bronze_ecommerce_sellers
     EXCEPT ALL
     SELECT _bronze_record_hash FROM silver_candidate_sellers
  )) AS bronze_rows_missing_in_candidate,
  (SELECT COUNT(*) FROM (
     SELECT _bronze_record_hash FROM silver_candidate_sellers
     EXCEPT ALL
     SELECT _record_hash FROM bronze_ecommerce_sellers
  )) AS unexpected_candidate_rows;

**Expected result:** both values are zero for every entity. This proves that the same
physical Bronze rows reached Candidate — not just an equal count.

### 8.2b Derived-field formula validation

Validate the documented Week-5 calculations without changing or filtering Candidate rows.


In [ ]:
%sql
WITH checks AS (
  SELECT
    (SELECT COUNT(*) FROM silver_candidate_order_items
     WHERE item_price IS NOT NULL AND quantity IS NOT NULL
       AND item_total <> CAST(item_price * quantity AS DECIMAL(12,2))) AS item_total_mismatches,
    (SELECT COUNT(*) FROM silver_candidate_order_items
     WHERE freight_value IS NOT NULL AND item_price IS NOT NULL AND quantity IS NOT NULL
       AND (item_price * quantity) <> 0
       AND freight_share <> CAST(freight_value / (item_price * quantity) AS DECIMAL(12,4))) AS freight_share_mismatches,
    (SELECT COUNT(*) FROM silver_candidate_orders
     WHERE purchase_ts IS NOT NULL AND delivered_ts IS NOT NULL
       AND delivery_days <> datediff(to_date(delivered_ts), to_date(purchase_ts))) AS delivery_days_mismatches,
    (SELECT COUNT(*) FROM silver_candidate_orders
     WHERE estimated_delivery_ts IS NOT NULL AND delivered_ts IS NOT NULL
       AND delay_days <> datediff(to_date(delivered_ts), to_date(estimated_delivery_ts))) AS delay_days_mismatches,
    (SELECT COUNT(*) FROM silver_candidate_orders
     WHERE estimated_delivery_ts IS NOT NULL AND delivered_ts IS NOT NULL
       AND delivery_band <> CASE
         WHEN datediff(to_date(delivered_ts), to_date(estimated_delivery_ts)) <= 0 THEN 'early_or_on_time'
         WHEN datediff(to_date(delivered_ts), to_date(estimated_delivery_ts)) BETWEEN 1 AND 3 THEN 'late_1_3_days'
         ELSE 'late_4_plus_days' END) AS delivery_band_mismatches
)
SELECT *,
       CASE WHEN item_total_mismatches + freight_share_mismatches + delivery_days_mismatches
                  + delay_days_mismatches + delivery_band_mismatches = 0
            THEN 'PASS' ELSE 'CHECK' END AS status
FROM checks;


### 8.3 Confirm Delta format for all five Candidate tables

In [ ]:
%sql
SELECT table_name, data_source_format
FROM system.information_schema.tables
WHERE table_catalog = current_catalog()
  AND table_schema = current_schema()
  AND table_name LIKE 'silver_candidate_%'
ORDER BY table_name;

# Part 9 — Controlled repeat-run test

The approved Week-5 exit requires a controlled rebuild with stable counts. This rerun recreates all five Candidate Delta tables from the already-built Candidate-ready views, then compares counts before and after. No rows are filtered, deduplicated or quarantined.


In [ ]:
%sql
-- Capture the pre-rerun Candidate counts.
CREATE OR REPLACE TEMP VIEW week5_rerun_before AS
SELECT 'orders' AS entity, COUNT(*) AS rows_before FROM silver_candidate_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM silver_candidate_order_items
UNION ALL SELECT 'payments', COUNT(*) FROM silver_candidate_payments
UNION ALL SELECT 'reviews', COUNT(*) FROM silver_candidate_reviews
UNION ALL SELECT 'sellers', COUNT(*) FROM silver_candidate_sellers;

-- Controlled rebuild of all five Candidate Delta tables.
CREATE OR REPLACE TABLE silver_candidate_orders USING DELTA AS SELECT * FROM orders_candidate_ready;
CREATE OR REPLACE TABLE silver_candidate_order_items USING DELTA AS SELECT * FROM order_items_candidate_ready;
CREATE OR REPLACE TABLE silver_candidate_payments USING DELTA AS SELECT * FROM payments_typed;
CREATE OR REPLACE TABLE silver_candidate_reviews USING DELTA AS SELECT * FROM reviews_typed;
CREATE OR REPLACE TABLE silver_candidate_sellers USING DELTA AS SELECT * FROM sellers_typed;

WITH after_run AS (
  SELECT 'orders' AS entity, COUNT(*) AS rows_after FROM silver_candidate_orders
  UNION ALL SELECT 'order_items', COUNT(*) FROM silver_candidate_order_items
  UNION ALL SELECT 'payments', COUNT(*) FROM silver_candidate_payments
  UNION ALL SELECT 'reviews', COUNT(*) FROM silver_candidate_reviews
  UNION ALL SELECT 'sellers', COUNT(*) FROM silver_candidate_sellers
)
SELECT b.entity, b.rows_before, a.rows_after, a.rows_after - b.rows_before AS difference,
       CASE WHEN a.rows_after = b.rows_before THEN 'PASS' ELSE 'CHECK' END AS status
FROM week5_rerun_before b
JOIN after_run a USING (entity)
ORDER BY entity;


### 9.2 Delta history after controlled rerun

Inspect the history of each Candidate table and retain the real execution output as Week-5 evidence.


In [ ]:
%sql
DESCRIBE HISTORY silver_candidate_orders;


In [ ]:
%sql
DESCRIBE HISTORY silver_candidate_order_items;


In [ ]:
%sql
DESCRIBE HISTORY silver_candidate_payments;


In [ ]:
%sql
DESCRIBE HISTORY silver_candidate_reviews;


In [ ]:
%sql
DESCRIBE HISTORY silver_candidate_sellers;


---

**Week-5 boundary:** stop here. Data Quality rule evaluation, pass/fail routing,
quarantine, Trusted Silver, Gold tables, Power BI and streaming belong to Week 6 and
later.